In [ ]:
import kagglehub
import pandas as pd
import numpy as np

# Download latest version
path = kagglehub.dataset_download("burak3ergun/loan-data-set")
print("Path to dataset files:", path)

# The csv inside is usually named loan_data_set.csv
import os
print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "loan_data_set.csv"))
df.head()


In [ ]:
# Basic shape and structure
print("Shape:", df.shape)
print(df.info())
print(df.describe())

# Check target distribution
print(df['Loan_Status'].value_counts())
print(df['Loan_Status'].value_counts(normalize=True) * 100)

# Check for missing values
print(df.isnull().sum())

# Check categorical columns
categorical_cols = df.select_dtypes(include='object').columns
print("Categorical columns:", categorical_cols.tolist())

for col in categorical_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Target distribution
sns.countplot(x='Loan_Status', data=df)
plt.title("Loan Status Distribution")
plt.show()

# Credit history vs loan status (typically the strongest predictor)
sns.countplot(x='Credit_History', hue='Loan_Status', data=df)
plt.title("Credit History vs Loan Status")
plt.show()

# Correlation among numeric features
plt.figure(figsize=(8,6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Drop Loan_ID — it's just an identifier, not a predictive feature
df.drop('Loan_ID', axis=1, inplace=True)

# Fill missing values
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['Married'].fillna(df['Married'].mode()[0], inplace=True)
df['Dependents'].fillna(df['Dependents'].mode()[0], inplace=True)
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0], inplace=True)
df['Credit_History'].fillna(df['Credit_History'].mode()[0], inplace=True)
df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0], inplace=True)
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)

# Clean Dependents column ("3+" -> 3)
df['Dependents'] = df['Dependents'].replace('3+', 3).astype(int)

# Encode categorical columns
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Loan_Status']
for col in binary_cols:
    df[col] = le.fit_transform(df[col])   # e.g. Loan_Status: N=0, Y=1

# Property_Area has 3 categories -> one-hot encode
df = pd.get_dummies(df, columns=['Property_Area'], drop_first=True)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics

model = GaussianNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

results = X_test.copy()
results['Actual'] = y_test.values
results['Predicted'] = y_pred
results['Actual_Label'] = results['Actual'].map({0: 'Not Fully Paid', 1: 'Fully Paid'})
results['Predicted_Label'] = results['Predicted'].map({0: 'Not Fully Paid', 1: 'Fully Paid'})

print(results[['Actual_Label', 'Predicted_Label']].head(10))